In [1]:
"""
Moteur d'indexation d'images — version Spark
=============================================
Migration de l'indexation séquentielle (pandas + boucle for) vers Spark.

Principe :
  - Chaque image = 1 partition Spark
  - cv2 s'exécute dans les workers via une UDF
  - Les histogrammes sont calculés en parallèle sur tous les cœurs
  - Le résultat est sauvegardé en CSV (compatible avec l'original) ET en Parquet

Usage :
    python3 image_indexer_spark.py --path /chemin/vers/images --methode HistGREY --bins 16
    python3 image_indexer_spark.py --path /chemin/vers/images --methode HistRGB  --bins 4
"""

import os
import sys
import argparse
import numpy as np
import pandas as pd
import cv2

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, pandas_udf
from pyspark.sql.types import ArrayType, FloatType, StringType
import pyspark.sql.functions as F

In [2]:
# ─────────────────────────────────────────────────────────────
# SESSION SPARK
# ─────────────────────────────────────────────────────────────

spark = (
    SparkSession.builder
    .appName("ImageIndexer")
    .master("local[*]")
    # Arrow accélère les échanges Python ↔ JVM pour les pandas_udf
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    # 1 partition par image = parallélisme maximal sur 10 000 images
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# ─────────────────────────────────────────────────────────────
# ÉTAPE 1 — Lister les images (remplace CreateFiles_base10000)
# ─────────────────────────────────────────────────────────────

def create_image_dataframe(path: str):
    """
    Remplace CreateFiles_base10000.
    Au lieu d'écrire un fichier texte, on crée directement un DataFrame Spark.
    Chaque ligne = un chemin d'image complet.
    """
    files = [
        os.path.join(path, f)
        for f in os.listdir(path)
        if not f.startswith(".")
        and f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tiff"))
    ]

    if not files:
        raise ValueError(f"Aucune image trouvée dans : {path}")

    print(f"  {len(files)} images trouvées")

    # On crée un DataFrame Spark avec 1 colonne : le chemin complet
    df = spark.createDataFrame([(f,) for f in files], ["filepath"])
    return df

In [ ]:
# ─────────────────────────────────────────────────────────────
# ÉTAPE 2 — UDFs de calcul d'histogramme
# ─────────────────────────────────────────────────────────────
# Pourquoi des UDF ?
# cv2 est une lib Python/C++ — Spark ne sait pas l'appeler directement.
# Une UDF (User Defined Function) permet d'exécuter du code Python
# sur chaque ligne d'un DataFrame Spark, en parallèle sur tous les workers.
# ─────────────────────────────────────────────────────────────

def _compute_gray_hist(filepath: str, bins: int) -> list:
    """
    Version interne de ComputeGrayLevelHistogram.
    Retourne une liste Python (sérialisable par Spark).
    """
    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if image is None:
        # Image illisible : on retourne un vecteur nul plutôt que planter le job
        return [0.0] * bins
    hist = cv2.calcHist([image], [0], None, [bins], [0, 256])
    hist = cv2.normalize(hist, hist)
    return hist.flatten().tolist()

In [ ]:
def _compute_rgb_hist(filepath: str, bins: int) -> list:
    """
    Version interne de ComputeRGBHistogram.
    bins^3 valeurs (ex : 4^3 = 64, 6^3 = 216).
    """
    image = cv2.imread(filepath, cv2.IMREAD_COLOR)
    if image is None:
        return [0.0] * (bins ** 3)
    b = [bins, bins, bins]
    hist = cv2.calcHist([image], [0, 1, 2], None, b, [0, 256, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist)
    return hist.flatten().tolist()


In [ ]:
def make_gray_udf(bins: int):
    """
    Fabrique une UDF Spark pour l'histogramme gris avec le nombre de bins fixé.
    On utilise une closure pour capturer bins (Spark ne passe pas de paramètres aux UDF).
    """
    @udf(returnType=ArrayType(FloatType()))
    def gray_udf(filepath: str):
        return _compute_gray_hist(filepath, bins)
    return gray_udf


In [ ]:
def make_rgb_udf(bins: int):
    """
    Fabrique une UDF Spark pour l'histogramme RGB avec le nombre de bins fixé.
    """
    @udf(returnType=ArrayType(FloatType()))
    def rgb_udf(filepath: str):
        return _compute_rgb_hist(filepath, bins)
    return rgb_udf

In [ ]:
# ─────────────────────────────────────────────────────────────
# ÉTAPE 3 — Indexation (remplace IndexDataBase)
# ─────────────────────────────────────────────────────────────

def index_database(path: str, methode: str, bins: int):
    """
    Remplace IndexDataBase.

    Différences clés vs l'original :
      - Plus de boucle for  → Spark distribue sur tous les cœurs
      - Plus de pd.concat() → coûteux et séquentiel
      - Sauvegarde en CSV (compatible) ET en Parquet (requêtable)

    Sorties :
      Base10000.HistGREY_16.csv     — compatible avec l'index original
      Base10000.HistGREY_16.parquet — format requêtable efficacement
    """

    print(f"\n  Méthode : {methode} | Bins : {bins}")

    # 1. Lister les images → DataFrame Spark
    df_images = create_image_dataframe(path)

    # 2. Choisir la bonne UDF selon la méthode
    if methode == "HistGREY":
        compute_hist = make_gray_udf(bins)
        expected_size = bins
    elif methode == "HistRGB":
        compute_hist = make_rgb_udf(bins)
        expected_size = bins ** 3
    else:
        sys.exit("methode doit être 'HistGREY' ou 'HistRGB'")

    print(f"  Taille du vecteur histogramme : {expected_size} valeurs")

    # 3. Appliquer l'UDF — Spark distribue le calcul sur tous les cœurs
    #    Chaque worker calcule l'histogramme de son lot d'images
    print("  Calcul des histogrammes en parallèle...")
    df_indexed = df_images.withColumn(
        "histogram",
        compute_hist(col("filepath"))
    )

    # 4. Exploser le vecteur en colonnes nommées h_0, h_1, ..., h_N
    #    (format compatible avec l'index original : 1 valeur par colonne)
    col_names = [f"h_{i}" for i in range(expected_size)]
    df_result = df_indexed.select(
        col("filepath"),
        *[col("histogram")[i].alias(col_names[i]) for i in range(expected_size)]
    )

    # 5. Sauvegarder
    base_name = f"Base10000.{methode}_{bins}"
    csv_path     = os.path.join(os.getcwd(), base_name + ".csv")
    parquet_path = os.path.join(os.getcwd(), base_name + ".parquet")

    print(f"  Sauvegarde CSV     → {csv_path}")
    (df_result
        .write
        .mode("overwrite")
        .option("header", "false")   # compatible avec l'original (pas d'en-tête)
        .csv(csv_path))

    print(f"  Sauvegarde Parquet → {parquet_path}")
    (df_result
        .write
        .mode("overwrite")
        .parquet(parquet_path))

    print(f"\n  Terminé — {df_result.count()} images indexées")
    return df_result

In [ ]:
# ─────────────────────────────────────────────────────────────
# POINT D'ENTRÉE
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Indexation d'images avec Spark")
    parser.add_argument("--path",    required=True,  help="Répertoire des images")
    parser.add_argument("--methode", required=True,  choices=["HistGREY", "HistRGB"])
    parser.add_argument("--bins",    required=True,  type=int,
                        help="HistGREY: 16, 64, 256 | HistRGB: 2, 4, 6")
    args = parser.parse_args()

    print("=" * 55)
    print("  Indexation d'images — Spark")
    print("=" * 55)

    df = index_database(
        path=args.path,
        methode=args.methode,
        bins=args.bins,
    )

    print("\n  Aperçu (5 premières lignes) :")
    df.show(5, truncate=True)

    spark.stop()